# Benchmark Suite — Model Benchmarking

Runs standard ML algorithms on all datasets in one or more collections.

**Prerequisites:**
1. The pipeline has been run with `benchmark-framework run ... --collection <id>`
2. The collection has been validated and exported (or at least processed)

**Usage:**
- Set `COLLECTIONS` to a list of collection IDs to benchmark specific ones
- Or leave it as `None` to benchmark all collections found in `data/processed/collections/`

In [ ]:
import sys
from pathlib import Path

# Make scripts importable for pipeline utilities
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer

import config

## Configuration

Set `COLLECTIONS` to a list of collection IDs, or leave as `None` to auto-discover
all collections in `data/processed/collections/`.

In [ ]:
# === CONFIG ===
# Set to a list of collection IDs, or None to auto-discover all
COLLECTIONS = None

if COLLECTIONS is None:
    # Auto-discover all collections in data/processed/collections/
    processed_root = config.PROCESSED_DIR
    if processed_root.exists():
        COLLECTIONS = sorted([d.name for d in processed_root.iterdir() if d.is_dir()])
    else:
        COLLECTIONS = []

if not COLLECTIONS:
    print('No processed collections found.')
    print(f'Looking in: {config.PROCESSED_DIR}')
    print('Run the pipeline first: benchmark-framework validate')
else:
    print(f'Will benchmark {len(COLLECTIONS)} collections:')
    for col in COLLECTIONS:
        ds_path = config.PROCESSED_DIR / col
        ds_count = len([d for d in ds_path.iterdir() if d.is_dir()]) if ds_path.exists() else 0
        print(f'  - {col} ({ds_count} datasets)')

## Helper Functions

These utilities handle:
- Detecting task type (classification vs regression) from the target column
- Building preprocessing pipelines that work with mixed numeric/categorical data
- Selecting appropriate algorithms and metrics per task type

In [ ]:
def detect_task_type(target_series: pd.Series, n_classes_threshold: int = 20) -> str:
    """Infer classification vs regression from a target Series."""
    n_unique = target_series.nunique()
    if n_unique <= n_classes_threshold:
        if pd.api.types.is_numeric_dtype(target_series):
            if n_unique <= 15 or (n_unique / len(target_series) < 0.05):
                return 'classification'
            return 'regression'
        return 'classification'
    return 'regression'


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Build a preprocessing ColumnTransformer for mixed-type data."""
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    transformers = []
    if numeric_cols:
        transformers.append((
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=50)),
            ]),
            categorical_cols,
        ))
    return ColumnTransformer(transformers)

In [ ]:
# Algorithms and cross-validation strategy per task type.
# Robust defaults that work across various dataset sizes and types.

CLASSIFIERS = {
    'LogisticRegression': LogisticRegression(
        max_iter=2000, C=1.0, solver='lbfgs', random_state=42,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        n_jobs=-1, random_state=42,
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.1,
        random_state=42,
    ),
}

REGRESSORS = {
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        n_jobs=-1, random_state=42,
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=150, max_depth=3, learning_rate=0.1,
        random_state=42,
    ),
}

def get_models_and_cv(task_type: str):
    """Return model dict and cross-validation splitter for a task type."""
    if task_type == 'classification':
        models = CLASSIFIERS
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'accuracy'
    else:
        models = REGRESSORS
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'neg_root_mean_squared_error'
    return models, cv, scoring

## Benchmark Runner

Runs all algorithms on all datasets in the specified collections.

In [ ]:
def benchmark_dataset(csv_path: Path, dataset_id: str, collection_id: str) -> dict:
    """Run all applicable algorithms on one dataset.

    Returns a dict with results per algorithm.
    For small datasets (<50 rows), GradientBoosting is skipped.
    """
    df = pd.read_csv(csv_path)

    # Target is the last column (pipeline convention)
    target_col = df.columns[-1]
    y = df[target_col]
    X = df.drop(columns=[target_col])

    task_type = detect_task_type(y)
    models, cv, scoring = get_models_and_cv(task_type)

    n_samples = len(df)
    results = {
        'collection_id': collection_id,
        'dataset_id': dataset_id,
        'task': task_type,
        'n_samples': n_samples,
        'n_features': X.shape[1],
    }

    preprocessor = build_preprocessor(X)

    for name, model in models.items():
        if n_samples < 50 and 'GradientBoosting' in name:
            results[name] = 'skipped (small dataset)'
            continue

        try:
            pipe = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier' if task_type == 'classification' else 'regressor', model),
            ])
            scores = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)

            if task_type == 'classification':
                results[name] = f'{scores.mean():.4f} +/- {scores.std():.4f}'
            else:
                rmse_scores = -scores
                results[name] = f'RMSE: {rmse_scores.mean():.4f} +/- {rmse_scores.std():.4f}'
        except Exception as e:
            results[name] = f'error: {type(e).__name__}: {e}'

    return results


def benchmark_collection(collection_id: str) -> list[dict]:
    """Run benchmarks on all datasets in a validated collection."""
    collection_path = config.PROCESSED_DIR / collection_id

    if not collection_path.exists():
        print(f'No processed data found for collection "{collection_id}"')
        return []

    dataset_dirs = sorted([d for d in collection_path.iterdir() if d.is_dir()])
    if not dataset_dirs:
        print(f'No datasets found in {collection_path}')
        return []

    all_results = []
    for ds_dir in dataset_dirs:
        csv_path = ds_dir / 'data.csv'
        if not csv_path.exists():
            continue

        print(f'  {collection_id}/{ds_dir.name}...')
        result = benchmark_dataset(csv_path, ds_dir.name, collection_id)
        all_results.append(result)

    return all_results

## Run Benchmarks

Benchmarks all collections listed in `COLLECTIONS` (set in Configuration cell).

In [ ]:
all_results = []
for collection_id in COLLECTIONS:
    print(f'\nBenchmarking collection: {collection_id}')
    results = benchmark_collection(collection_id)
    all_results.extend(results)

if all_results:
    results_df = pd.DataFrame(all_results)
    priority_cols = ['collection_id', 'dataset_id', 'task', 'n_samples', 'n_features']
    col_order = priority_cols + [c for c in results_df.columns if c not in priority_cols]
    display(results_df[col_order].style.hide(axis='index'))

    print(f'\nBenchmarked {len(all_results)} datasets across {len(COLLECTIONS)} collections')
else:
    print('No results to display.')

## Save Results

Persist all benchmark results to a single CSV.

In [ ]:
if all_results:
    output_path = config.EXPORTS_DIR / 'all_collections_benchmark_results.csv'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_results).to_csv(output_path, index=False)
    print(f'Results saved to: {output_path}')